# Iris cross validation

In [1]:
import numpy as np
import pandas as pd
import keras

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

import tensorflow as tf

In [2]:
tf.__version__

'2.13.0'

In [3]:
seed = 5
np.random.seed(seed)

In [4]:
iris = pd.read_csv("..\input\Iris.csv")
dataset = iris.values
X = dataset[:,1:5].astype(float)
Y = dataset[:,5]
iris.shape

(150, 6)

In [5]:
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
inv_encoded_Y = encoder.inverse_transform(encoded_Y)
answer_Y = tf.keras.utils.to_categorical(encoded_Y)

In [6]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(8, input_dim=4, activation=tf.nn.relu))
model.add(tf.keras.layers.Dense(3, activation=tf.nn.softmax))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [7]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 8)                 40        
                                                                 
 dense_1 (Dense)             (None, 3)                 27        
                                                                 
Total params: 67 (268.00 Byte)
Trainable params: 67 (268.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
Fold_number = 10

In [9]:
kf = KFold(n_splits=Fold_number,random_state=12, shuffle=True)
kf.get_n_splits(X)
print(kf)  

KFold(n_splits=10, random_state=12, shuffle=True)


In [10]:
for train_index, test_index in kf.split(X):
    print("TRAIN:", train_index, "\n\nTEST:", test_index, '\n\n')

TRAIN: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  17  18
  19  20  22  24  25  26  27  28  29  30  32  33  34  35  36  37  41  42
  43  44  45  46  47  48  49  51  52  53  54  55  56  57  58  59  60  61
  62  63  64  65  67  68  69  70  71  72  73  74  75  76  77  78  79  80
  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98
 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 117 118
 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 136 137
 138 139 140 141 142 144 145 147 149] 

TEST: [ 16  21  23  31  38  39  40  50  66  99 116 135 143 146 148] 


TRAIN: [  0   2   3   4   5   7   8   9  10  12  13  14  15  16  17  18  19  20
  21  22  23  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39
  40  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58
  59  60  62  63  64  65  66  67  68  69  70  73  74  75  76  79  80  81
  82  83  84  85  86  87  88  89  91  92  93  94  95  96  97  98  99 100

In [11]:
scores = []
i = 0
for train_index, test_index in kf.split(X):
    i+=1
    
    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = answer_Y[train_index], answer_Y[test_index]
    model.fit(X_train,Y_train,epochs=100,batch_size=5,verbose=0)
    score_train = model.evaluate(X_train,Y_train,verbose=0) 
    score_test = model.evaluate(X_test,Y_test,verbose=0) 
    print("Processing fold {:d},".format(i),"train:{:0.2f},".format\
          (score_train[1]),"test:{:0.2f}".format(score_test[1]))
    scores.append(score_test[1])

Processing fold 1, train:0.96, test:1.00
Processing fold 2, train:0.97, test:0.93
Processing fold 3, train:0.96, test:1.00
Processing fold 4, train:0.98, test:1.00
Processing fold 5, train:0.99, test:0.93
Processing fold 6, train:0.98, test:1.00
Processing fold 7, train:0.99, test:0.93
Processing fold 8, train:0.98, test:1.00
Processing fold 9, train:0.98, test:1.00
Processing fold 10, train:0.99, test:0.93


In [12]:
print("Cross Validation with {:d}".format(Fold_number),\
      "folds.\nCV score = {:0.3f} +/- {:0.3f}".\
      format(np.mean(np.array(scores)),\
             np.std(np.array(scores))))

Cross Validation with 10 folds.
CV score = 0.973 +/- 0.033
